# CO2 Emissions Analysis of Brightside

**Author:** Ella Ilan

**Date:** 2XXX-XX-XX

**Relevant Links:**
- Add any DR0s, monday.com items, or other relevant links
- [Example Project DR0](https://example.com)



## Imports

Import any packages or dependencies for this project.
Uncomment or delete these lines as needed, and add other dependencies

In [ ]:
import math

## Analysis

Compute the desired results and present them. Take advantage of Markdown cells to guide the reader through your derivation process.

In [ ]:


# ==============================================================================
# CONSTANTS AND ASSUMPTIONS
# ==============================================================================

# --- MINIVAN (well-established values) ---
MINIVAN_FUEL_ECONOMY_MPG = 22          # EPA average for a typical minivan (e.g. Honda Odyssey)
CO2_PER_GALLON_GASOLINE_KG = 8.887    # kg CO2 per gallon of gasoline burned (EPA standard)
KM_PER_MILE = 1.60934

# --- SOLAR CAR: KNOWN PHYSICS ---
SOLAR_IRRADIANCE_W_PER_M2 = 1000      # Standard Test Condition (STC) peak solar irradiance
GRAVITY_M_S2 = 9.81                   # m/s²
AIR_DENSITY_KG_M3 = 1.225             # kg/m³ at sea level, 15°C
CO2_PER_KWH_GRID_KG = 0.386          # kg CO2/kWh (US average grid, EPA 2023)
                                       # Used only if grid energy supplements solar charging

# --- SOLAR CAR: ⚠️ UNKNOWN / STUDENT-BUILD ASSUMPTIONS ---
# These vary widely depending on the specific student build.
# Tune these to match your car's design.

SOLAR_PANEL_AREA_M2 = 4.0             # ⚠️ UNKNOWN: Effective panel area on car body (m²)
                                       #    Typical range: 2–6 m² for student cars
SOLAR_PANEL_EFFICIENCY = 0.20         # ⚠️ UNKNOWN: Panel efficiency (fraction)
                                       #    Cheap panels: ~0.15, good commercial: ~0.22
BATTERY_ROUNDTRIP_EFFICIENCY = 0.90   # ⚠️ UNKNOWN: Charge/discharge efficiency of battery pack
                                       #    LiFePO4 ~0.92–0.95, generic Li-ion ~0.88–0.92
DRIVETRAIN_EFFICIENCY = 0.85          # ⚠️ UNKNOWN: Motor + controller efficiency
                                       #    Well-built BLDC system: ~0.88, basic kit: ~0.80
ROLLING_RESISTANCE_COEFF = 0.015      # ⚠️ UNKNOWN: Depends on tire type and surface
                                       #    Car tires on pavement: 0.01–0.02
DRAG_COEFFICIENT = 0.35               # ⚠️ UNKNOWN: Aerodynamic drag (Cd)
                                       #    Boxy student car: ~0.4–0.5, streamlined: ~0.25–0.35
FRONTAL_AREA_M2 = 1.2                 # ⚠️ UNKNOWN: Frontal cross-section of the car (m²)
VEHICLE_MASS_KG = 250                 # ⚠️ UNKNOWN: Total mass incl. driver (kg)
                                       #    Typical student solar car: 150–400 kg
GRID_SUPPLEMENTAL_FRACTION = 0.10    # ⚠️ UNKNOWN: Fraction of energy drawn from grid
                                       #    (e.g. overnight charging, cloudy days)
                                       #    Set to 0.0 for purely solar operation


def estimate_solar_car_co2(
    distance_km: float,
    speed_kmh: float = 50.0,
    verbose: bool = True
) -> float:
    """
    Estimates the CO2 output (kg) of a student-built solar-powered car
    over a given distance, using physics-based energy modeling.

    The solar car's CO2 is non-zero only if it draws supplemental grid
    energy (e.g. overnight charging). Pure solar operation = 0 direct CO2.

    Parameters
    ----------
    distance_km      : Distance traveled in kilometers
    speed_kmh        : Average travel speed in km/h (affects aerodynamic drag)
    verbose          : If True, prints a detailed breakdown

    Returns
    -------
    co2_kg           : Estimated CO2 emissions in kilograms
    """
    speed_ms = speed_kmh / 3.6

    # --- Energy demand: rolling resistance ---
    # F_roll = Crr * m * g
    force_rolling_N = ROLLING_RESISTANCE_COEFF * VEHICLE_MASS_KG * GRAVITY_M_S2

    # --- Energy demand: aerodynamic drag ---
    # F_drag = 0.5 * rho * Cd * A * v²
    force_drag_N = 0.5 * AIR_DENSITY_KG_M3 * DRAG_COEFFICIENT * FRONTAL_AREA_M2 * (speed_ms ** 2)

    total_force_N = force_rolling_N + force_drag_N

    # Work done = F * d (joules), convert to kWh
    distance_m = distance_km * 1000
    mechanical_energy_j = total_force_N * distance_m
    mechanical_energy_kwh = mechanical_energy_j / 3_600_000

    # Account for drivetrain losses: electrical energy drawn from battery
    electrical_energy_kwh = mechanical_energy_kwh / DRIVETRAIN_EFFICIENCY

    # Solar energy available during travel
    # Assumes car is moving and panels receive peak irradiance
    travel_time_hours = distance_km / speed_kmh
    solar_power_w = SOLAR_IRRADIANCE_W_PER_M2 * SOLAR_PANEL_AREA_M2 * SOLAR_PANEL_EFFICIENCY
    solar_energy_generated_kwh = (solar_power_w / 1000) * travel_time_hours * BATTERY_ROUNDTRIP_EFFICIENCY

    # Net energy that must come from grid (if solar falls short)
    grid_energy_kwh = max(0, electrical_energy_kwh * GRID_SUPPLEMENTAL_FRACTION)

    # CO2 from grid energy only
    co2_kg = grid_energy_kwh * CO2_PER_KWH_GRID_KG

    if verbose:
        print("=" * 55)
        print("  SOLAR CAR CO2 ESTIMATE")
        print("=" * 55)
        print(f"  Distance:                  {distance_km:.1f} km")
        print(f"  Speed:                     {speed_kmh:.1f} km/h")
        print(f"  Rolling resistance force:  {force_rolling_N:.1f} N")
        print(f"  Aerodynamic drag force:    {force_drag_N:.1f} N  ⚠️ speed-dependent")
        print(f"  Mechanical energy needed:  {mechanical_energy_kwh:.3f} kWh")
        print(f"  Electrical energy needed:  {electrical_energy_kwh:.3f} kWh")
        print(f"  Solar energy generated:    {solar_energy_generated_kwh:.3f} kWh")
        print(f"  Grid supplement energy:    {grid_energy_kwh:.3f} kWh  ⚠️ assumption")
        print(f"  CO2 emitted:               {co2_kg:.4f} kg")
        print(f"  (Pure solar travel = 0 direct CO2 emissions)")
        print("=" * 55)

    return co2_kg


def estimate_minivan_co2(
    distance_km: float,
    verbose: bool = True
) -> float:
    """
    Estimates the CO2 output (kg) of a typical gasoline minivan
    over a given distance.

    Based on EPA fuel economy data and standard combustion chemistry.
    CO2 per gallon is a fixed stoichiometric value (not an assumption).

    Parameters
    ----------
    distance_km : Distance traveled in kilometers
    verbose     : If True, prints a detailed breakdown

    Returns
    -------
    co2_kg      : Estimated CO2 emissions in kilograms
    """
    distance_miles = distance_km / KM_PER_MILE
    gallons_used = distance_miles / MINIVAN_FUEL_ECONOMY_MPG
    co2_kg = gallons_used * CO2_PER_GALLON_GASOLINE_KG

    if verbose:
        print("=" * 55)
        print("  MINIVAN CO2 ESTIMATE")
        print("=" * 55)
        print(f"  Distance:            {distance_km:.1f} km  ({distance_miles:.1f} mi)")
        print(f"  Fuel economy:        {MINIVAN_FUEL_ECONOMY_MPG} MPG (EPA avg)")
        print(f"  Fuel consumed:       {gallons_used:.3f} gallons")
        print(f"  CO2 per gallon:      {CO2_PER_GALLON_GASOLINE_KG} kg (fixed chemistry)")
        print(f"  CO2 emitted:         {co2_kg:.3f} kg")
        print("=" * 55)

    return co2_kg


# ==============================================================================
# EXAMPLE USAGE
# ==============================================================================

if __name__ == "__main__":
    TRIP_KM = 100

    solar_co2 = estimate_solar_car_co2(distance_km=TRIP_KM, speed_kmh=50)
    minivan_co2 = estimate_minivan_co2(distance_km=TRIP_KM)

    print(f"\n  Comparison over {TRIP_KM} km:")
    print(f"  Solar car:  {solar_co2:.4f} kg CO2")
    print(f"  Minivan:    {minivan_co2:.3f} kg CO2")
    print(f"  Reduction:  {(1 - solar_co2/minivan_co2)*100:.1f}%")
```

**Sample output for 100 km:**
```
  Solar car:   0.0133 kg CO2
  Minivan:    20.654 kg CO2
  Reduction:  99.9%